In [0]:
%pip install -q -r requirements.txt

In [0]:
from neo4j import GraphDatabase
import os
from dotenv import load_dotenv

In [0]:
# Load environment variables and get credentials
load_dotenv()
URI = os.getenv('NEO4J_URI')
USER = os.getenv('NEO4J_USER')
PASSWORD = os.getenv('NEO4J_PASSWORD')

# Connect to neo4j database
AUTH = (USER, PASSWORD)
driver = GraphDatabase.driver(URI, auth=AUTH)

In [0]:
# Cypher query to create nodes and relationships
cypher_script = """
// 1. Hardwares
MERGE (fbs:Component {name: 'Flow Battery System'})
MERGE (bms:Component {name: 'Battery Management System'})
MERGE (cooling:Component {name: 'Cooling System'})
MERGE (pipe:Component {name: 'Pipe'})
MERGE (tank_c:Component {name: 'Tank_Catholyte', size: '150L'})
MERGE (tank_a:Component {name: 'Tank_Anolyte', size: '150L'})
MERGE (pump:Component {name: 'Pump'})
MERGE (temp_sensor:Component {name: 'Temperature Sensor'})
MERGE (flow_sensor:Component {name: 'Flow Sensor'})
MERGE (stack:Component {name: 'Battery Stack', amount: '40 cells'})
MERGE (cell:Component {name: 'Unit Cell'})
MERGE (electrode:Component {name: 'Electrode'})
MERGE (membrane:Component {name: 'Membrane & Separator'})
MERGE (bipolar:Component {name: 'Bipolar Plate'})
MERGE (frame:Component {name: 'Frame'})

MERGE (fbs)-[:CONTAINS]->(bms)
MERGE (fbs)-[:CONTAINS]->(cooling)
MERGE (fbs)-[:CONTAINS]->(pipe)
MERGE (fbs)-[:CONTAINS]->(tank_c)
MERGE (fbs)-[:CONTAINS]->(tank_a)
MERGE (fbs)-[:CONTAINS]->(pump)
MERGE (fbs)-[:CONTAINS]->(temp_sensor)
MERGE (fbs)-[:CONTAINS]->(flow_sensor)
MERGE (fbs)-[:CONTAINS]->(stack)
MERGE (stack)-[:CONTAINS]->(cell)
MERGE (cell)-[:CONTAINS]->(electrode)
MERGE (cell)-[:CONTAINS]->(membrane)
MERGE (cell)-[:CONTAINS]->(bipolar)
MERGE (cell)-[:CONTAINS]->(frame)

// 2. Material Subcategory
MERGE (carbon_paper:Material {name: 'Carbon Paper'})
MERGE (carbon_felt:Material {name: 'Carbon Felt'})
MERGE (separator:Material {name: 'Separator'})
MERGE (aem:Material {name: 'AEM'})
MERGE (cem:Material {name: 'CEM'})
MERGE (carbon_bp:Material {name: 'Carbon Bipolar Plate'})
MERGE (carbon_poly:Material {name: 'Carbon + Polymer'})

MERGE (carbon_paper)-[:IS_A]->(electrode)
MERGE (carbon_felt)-[:IS_A]->(electrode)
MERGE (separator)-[:IS_A]->(membrane)
MERGE (aem)-[:IS_A]->(membrane)
MERGE (cem)-[:IS_A]->(membrane)
MERGE (carbon_bp)-[:IS_A]->(bipolar)
MERGE (carbon_poly)-[:IS_A]->(bipolar)

// 3. Material Properties
MERGE (low_frame_thk:Property {name: 'Low Frame Thickness'})
MERGE (high_frame_thk:Property {name: 'High Frame Thickness'})
MERGE (high_mem_pore:Property {name: 'High Membrane Pore Size'})
MERGE (low_mem_pore:Property {name: 'Low Membrane Pore Size'})
MERGE (high_mem_thk:Property {name: 'High Membrane Thickness'})
MERGE (low_mem_thk:Property {name: 'Low Membrane Thickness'})

MERGE (frame)-[:HAS_PROPERTIES]->(low_frame_thk)
MERGE (frame)-[:HAS_PROPERTIES]->(high_frame_thk)
MERGE (membrane)-[:HAS_PROPERTIES]->(high_mem_pore)
MERGE (membrane)-[:HAS_PROPERTIES]->(low_mem_pore)
MERGE (membrane)-[:HAS_PROPERTIES]->(high_mem_thk)
MERGE (membrane)-[:HAS_PROPERTIES]->(low_mem_thk)

// 4. Chemicals, Health, SOC
MERGE (electrolyte_a:Chemical {name: 'Electrolyte Anolyte', volume: '120L'})
MERGE (electrolyte_c:Chemical {name: 'Electrolyte Catholyte', volume: '120L'})
MERGE (reducing_agent:Chemical {name: 'Reducing Agent'})
MERGE (v2:Chemical {name: 'V2+'})
MERGE (v3:Chemical {name: 'V3+'})
MERGE (v4:Chemical {name: 'V4+'})
MERGE (v5:Chemical {name: 'V5+'})

MERGE (tank_a)-[:USES]->(electrolyte_a)
MERGE (tank_c)-[:USES]->(electrolyte_c)

MERGE (h_fresh_a:HealthState {name: 'Anolyte Fresh Oxidation State', ox_state: 3.5})
MERGE (h_aged_a:HealthState {name: 'Anolyte Aged Oxidation State', ox_state: 3.6})
MERGE (chargeState_fresh_a:chargeState {name: 'Anolyte Fresh Charge State'})
MERGE (chargeState_aged_a:chargeState {name: 'Anolyte Aged Charge State'})
MERGE (electrolyte_a)-[:CURRENT_HEALTH]->(h_fresh_a)
MERGE (electrolyte_a)-[:CURRENT_HEALTH]->(h_aged_a)
MERGE (h_fresh_a)-[:DEGRADES_TO]->(h_aged_a)
MERGE (h_aged_a)-[:RESTORED_TO]->(h_fresh_a)
MERGE (h_fresh_a)-[:HAS_CONDITION]->(chargeState_fresh_a)
MERGE (h_aged_a)-[:HAS_CONDITION]->(chargeState_aged_a)

MERGE (h_fresh_c:HealthState {name: 'Catholyte Fresh State', ox_state: 3.5})
MERGE (h_aged_c:HealthState {name: 'Catholyte Aged State', ox_state: 3.6})
MERGE (chargeState_fresh_c:chargeState {name: 'Catholyte Fresh Charge State'})
MERGE (chargeState_aged_c:chargeState {name: 'Catholyte Aged Charge State'})
MERGE (electrolyte_c)-[:CURRENT_HEALTH]->(h_fresh_c)
MERGE (electrolyte_c)-[:CURRENT_HEALTH]->(h_aged_c)
MERGE (h_fresh_c)-[:DEGRADES_TO]->(h_aged_c)
MERGE (h_aged_c)-[:RESTORED_TO]->(h_fresh_c)
MERGE (h_fresh_c)-[:HAS_CONDITION]->(chargeState_fresh_c)
MERGE (h_aged_c)-[:HAS_CONDITION]->(chargeState_aged_c)
MERGE (reducing_agent)-[:ENABLES_RESTORATION]->(h_aged_c)

// Electrolyte shift
MERGE (chargeState_fresh_a)-[:CONTAINS {soc: '50%', percentage: '50%'}]->(v3)
MERGE (chargeState_fresh_a)-[:CONTAINS {soc: '50%', percentage: '50%'}]->(v2)
MERGE (chargeState_aged_a)-[:CONTAINS {soc: '50%', percentage: '60%'}]->(v3)
MERGE (chargeState_aged_a)-[:CONTAINS {soc: '50%', percentage: '40%'}]->(v2)

MERGE (chargeState_fresh_c)-[:CONTAINS {soc: '50%', percentage: '50%'}]->(v5)
MERGE (chargeState_fresh_c)-[:CONTAINS {soc: '50%', percentage: '50%'}]->(v4)
MERGE (chargeState_aged_c)-[:CONTAINS {soc: '50%', percentage: '60%'}]->(v5)
MERGE (chargeState_aged_c)-[:CONTAINS {soc: '50%', percentage: '40%'}]->(v4)

// Vanadium reactions
MERGE (v2)-[:OXIDIZED_TO]->(v3)
MERGE (v3)-[:OXIDIZED_TO]->(v4)
MERGE (v4)-[:OXIDIZED_TO]->(v5)
MERGE (v3)-[:REDUCED_TO]->(v2)
MERGE (v4)-[:REDUCED_TO]->(v3)
MERGE (v5)-[:REDUCED_TO]->(v4)

// 5. Mechanisms & Failure Events
// Carbon and high voltage
MERGE (high_voltage:OperationalCondition {name: 'High Voltage'})
MERGE (carbon_corrosion:Mechanism {name: 'Carbon Corrosion'})
MERGE (carbon_felt)-[:EXPERIENCES]->(high_voltage)
MERGE (high_voltage)-[:CAUSES]->(carbon_corrosion)

// Compression and Flow
MERGE (high_comp:Mechanism {name: 'High Compression'})
MERGE (low_comp:Mechanism {name: 'Low Compression'})
MERGE (high_flow_res:Mechanism {name: 'High Flow Resistance'})
MERGE (low_flow_res:Mechanism {name: 'Low Flow Resistance'})
MERGE (low_flowrate:Mechanism {name: 'Low Flowrate'})
MERGE (high_flowrate:Mechanism {name: 'High Flowrate'})

MERGE (low_frame_thk)-[:CAUSES]->(high_comp)
MERGE (high_frame_thk)-[:CAUSES]->(low_comp)
MERGE (high_comp)-[:CAUSES]->(high_flow_res)
MERGE (low_comp)-[:CAUSES]->(low_flow_res)
MERGE (high_flow_res)-[:CAUSES]->(low_flowrate)
MERGE (low_flow_res)-[:CAUSES]->(high_flowrate)

// Membrane effects
MERGE (high_crossover:Mechanism {name: 'High Crossover'})
MERGE (low_crossover:Mechanism {name: 'Low Crossover'})
MERGE (high_ir:Mechanism {name: 'High Internal Resistance'})
MERGE (low_ir:Mechanism {name: 'Low Internal Resistance'})

MERGE (high_mem_pore)-[:CAUSES]->(high_crossover)
MERGE (low_mem_pore)-[:CAUSES]->(low_crossover)
MERGE (high_mem_thk)-[:CAUSES]->(high_ir)
MERGE (high_mem_thk)-[:CAUSES]->(low_crossover) 
MERGE (low_mem_thk)-[:CAUSES]->(low_ir)
MERGE (low_mem_thk)-[:CAUSES]->(high_crossover)

MERGE (low_flowrate)-[:CAUSES]->(high_ir)
MERGE (high_flowrate)-[:CAUSES]->(low_ir)

// Gas effects
MERGE (h2_evolution:Mechanism {name: 'Hydrogen Evolution'})
MERGE (bubble:Mechanism {name: 'Bubble Formation'})
MERGE (masking:Mechanism {name: 'Electrode Masking'})
MERGE (blockage:Mechanism {name: 'Flow Blockage'})

MERGE (v2)-[:CAUSES]->(h2_evolution)
MERGE (h2_evolution)-[:CAUSES]->(bubble)
MERGE (bubble)-[:CAUSES]->(masking)
MERGE (masking)-[:CAUSES]->(high_ir)
MERGE (bubble)-[:CAUSES]->(blockage)
MERGE (blockage)-[:CAUSES]->(high_flow_res)

// Precipitation effects
MERGE (v5_precip:Mechanism {name: 'V5+ Precipitation'})
MERGE (exothermic_reaction:Mechanism {name: 'Exothermic Reaction'})
MERGE (v5)-[:CAUSES]->(v5_precip)
MERGE (v5_precip)-[:CAUSES]->(blockage)
MERGE (exothermic_reaction)-[:SPEEDS_UP]->(v5_precip)

// Leakages
MERGE (screws:Mechanism {name: 'Screws Loose'})
MERGE (leakage:Event {name: 'Leakage'})
MERGE (mem_burst:Event {name: 'Membrane Burst'})
MERGE (mixing:Mechanism {name: 'Electrolyte Mixing'})
MERGE (env_contam:Event {name: 'Environment Contamination'})
MERGE (short_circuit:Event {name: 'Short Circuit'})
MERGE (safety_issue:Event {name: 'Safety Issue'})

MERGE (screws)-[:CAUSES]->(leakage)
MERGE (high_flow_res)-[:CAUSES]->(leakage)
MERGE (high_flow_res)-[:CAUSES]->(mem_burst)
MERGE (mem_burst)-[:CAUSES]->(mixing)
MERGE (mixing)-[:CAUSES]->(exothermic_reaction)
MERGE (leakage)-[:CAUSES]->(env_contam)
MERGE (leakage)-[:CAUSES]->(short_circuit)
MERGE (short_circuit)-[:CAUSES]->(exothermic_reaction)
MERGE (short_circuit)-[:CAUSES]->(safety_issue)

// 6. Metrics & Efficiency Trade-offs
MERGE (low_ce:Metric {name: 'Low Coulombic Efficiency'})
MERGE (high_ce:Metric {name: 'High Coulombic Efficiency'})
MERGE (low_ve:Metric {name: 'Low Voltage Efficiency'})
MERGE (high_ve:Metric {name: 'High Voltage Efficiency'})
MERGE (low_cap:Metric {name: 'Low Capacity'})
MERGE (pump_power:Metric {name: 'High Pump Power Required'})
MERGE (round_trip:Metric {name: 'Round-Trip Energy Efficiency Trade-off'})

MERGE (high_crossover)-[:LEADS_TO]->(low_ce)
MERGE (low_crossover)-[:LEADS_TO]->(high_ce)
MERGE (high_ir)-[:LEADS_TO]->(low_ve)
MERGE (low_ir)-[:LEADS_TO]->(high_ve)
MERGE (high_crossover)-[:LEADS_TO]->(low_cap)
MERGE (leakage)-[:LEADS_TO]->(low_cap)
MERGE (carbon_corrosion)-[:CAUSES]->(high_ir)
MERGE (carbon_corrosion)-[:CAUSES]->(blockage)
MERGE (high_flow_res)-[:CAUSES]->(pump_power)

MERGE (low_ce)-[:HAS_IMPACT_ON]->(round_trip)
MERGE (high_ce)-[:HAS_IMPACT_ON]->(round_trip)
MERGE (low_ve)-[:HAS_IMPACT_ON]->(round_trip)
MERGE (high_ve)-[:HAS_IMPACT_ON]->(round_trip)
MERGE (pump_power)-[:HAS_IMPACT_ON]->(round_trip)
"""

In [0]:
def build_graph():
  with driver.session() as session:
    session.run(cypher_script)

if __name__ == "__main__":
  build_graph()